# Credit-Card Fraud Detection — Sequential (Time-Series) Formulation

This notebook frames fraud detection as a *sequential* problem: given the last **20 transactions of a card** (in time order), predict whether the **most recent** transaction is fraudulent, using an **LSTM**.

The other notebooks frame the same data as a *tabular, point-wise* classification problem (tree/boosting models). **The two use different data units and evaluation protocols and are therefore not directly comparable** — see the closing notes.

**Protocol here is strictly temporal** (train on the past, evaluate on the future), because a sliding-window formulation with a random split leaks (see "Pitfalls addressed").

Dataset: [credit-card-transactions-dataset](https://www.kaggle.com/datasets/priyamchoksi/credit-card-transactions-dataset) (Sparkov-style simulator).


## Imports & Random Seed

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, precision_recall_curve, classification_report,
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

## Load data

In [2]:
CSV_PATH = "/kaggle/input/datasets/priyamchoksi/credit-card-transactions-dataset/credit_card_transactions.csv"

df = pd.read_csv(CSV_PATH)
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
print(df.shape)
print(f"Fraud rate: {df['is_fraud'].mean():.4%}")

(1296675, 24)
Fraud rate: 0.5789%


## Feature engineering

- **Time parts** from the transaction timestamp (hour, weekday, month, ...): cyclical
  signals that generalise across time.
- **Geographic distance** between the card-holder's home and the merchant, computed with
  the **haversine formula** (vectorised). We do *not* use plain Euclidean distance on
  (lat, long): one degree of longitude shrinks with latitude, so Euclidean would be a
  systematically biased, location-dependent approximation.
- **Card-holder age** from date of birth.

`unix_time` is kept only for ordering/splitting and is **excluded from model features**:
under a temporal split every test `unix_time` is greater than every train value, so feeding
it in would be a look-ahead leak. `trans_year` is excluded for the same reason.

In [3]:
ts = df["trans_date_trans_time"].dt
df["trans_year"]    = ts.year          # used to compute age; NOT a model feature
df["trans_month"]   = ts.month
df["trans_day"]     = ts.day
df["trans_season"]  = ts.month % 12 // 3 + 1   # 1=Winter 2=Spring 3=Summer 4=Autumn
df["trans_weekday"] = ts.weekday
df["trans_hour"]    = ts.hour
df["trans_minute"]  = ts.minute
df["trans_second"]  = ts.second

# --- haversine distance (vectorised, km) ---
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))

df["distance"] = haversine_km(df["lat"], df["long"], df["merch_lat"], df["merch_long"])

# --- card-holder age ---
df["dob"] = pd.to_datetime(df["dob"])
df["card_holder_age"] = df["trans_year"] - df["dob"].dt.year

## Configuration

In [4]:
SEQUENCE_LENGTH = 20
GROUP_COL = "cc_num"        # one sequence timeline per card
TIME_COL  = "unix_time"     # ordering / splitting only -- NOT a model feature
TARGET    = "is_fraud"

CATEGORICAL_FEATURES = ["category", "gender", "state"]
NUMERIC_FEATURES = [
    "amt", "lat", "long", "city_pop", "merch_lat", "merch_long",
    "trans_month", "trans_day", "trans_season", "trans_weekday",
    "trans_hour", "trans_minute", "trans_second", "distance", "card_holder_age",
]  # NOTE: 'unix_time' and 'trans_year' intentionally excluded 

TRAIN_Q, VAL_Q = 0.70, 0.80   # time quantiles: <=q70 train | (q70,q80] val | >q80 test

keep = [GROUP_COL, TIME_COL, TARGET] + NUMERIC_FEATURES + CATEGORICAL_FEATURES
df = df[keep].sort_values(TIME_COL).reset_index(drop=True)

## Temporal split

We split rows by **time quantiles** (no shuffling), then build sequences *within* each
split. This mimics deployment — train on the past, evaluate on the future — and, crucially,
guarantees that no sliding window straddles a split boundary.

In [5]:
t = df[TIME_COL].values
q_train, q_val = np.quantile(t, TRAIN_Q), np.quantile(t, VAL_Q)

train_df = df[df[TIME_COL] <= q_train].copy()
val_df   = df[(df[TIME_COL] > q_train) & (df[TIME_COL] <= q_val)].copy()
test_df  = df[df[TIME_COL] > q_val].copy()

print(f"rows  -> train:{len(train_df):>8}  val:{len(val_df):>8}  test:{len(test_df):>8}")
print(f"fraud -> train:{int(train_df[TARGET].sum()):>8}  "
      f"val:{int(val_df[TARGET].sum()):>8}  test:{int(test_df[TARGET].sum()):>8}")

rows  -> train:  907672  val:  129668  test:  259335
fraud -> train:    5121  val:     847  test:    1538


## Preprocessing & sequence construction

- **One-hot** for categoricals (no false ordinal ranking), **standard-scale** for numerics.
- The preprocessor is **fit on train only**, then applied to val/test (no statistics leak).
- Within each split, rows are sorted by `(card, time)` and windows are built per card. A
  window's label is the fraud flag of its **last** transaction.

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        # sklearn >=1.2 uses sparse_output; for older versions use sparse=False
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)


# Sliding windows per card. `frame` must be sorted by [GROUP_COL, TIME_COL]
# and aligned row-for-row with `feature_matrix`. A window's label is the fraud
# flag of its LAST transaction.
def create_sequences(frame, feature_matrix, seq_length):
    targets = frame[TARGET].values
    Xs, ys = [], []
    for _, pos in frame.groupby(GROUP_COL, sort=False).indices.items():
        pos = np.asarray(pos)
        g_feats, g_targets = feature_matrix[pos], targets[pos]
        for i in range(len(pos) - seq_length + 1):
            Xs.append(g_feats[i:i + seq_length])
            ys.append(g_targets[i + seq_length - 1])
    if not Xs:
        return np.empty((0, seq_length, feature_matrix.shape[1]), dtype="float32"), \
               np.empty((0,), dtype="int8")
    return np.asarray(Xs, dtype="float32"), np.asarray(ys, dtype="int8")


def prepare_split(frame, fit):
    frame = frame.sort_values([GROUP_COL, TIME_COL]).reset_index(drop=True)  # sort BEFORE windowing
    feats = preprocessor.fit_transform(frame) if fit else preprocessor.transform(frame)
    feats = np.asarray(feats, dtype="float32")
    return create_sequences(frame, feats, SEQUENCE_LENGTH)

In [7]:
X_train, y_train = prepare_split(train_df, fit=True)    
X_val,   y_val   = prepare_split(val_df,   fit=False)
X_test,  y_test  = prepare_split(test_df,  fit=False)

num_features = X_train.shape[2]
print(f"seq   -> train:{X_train.shape}  val:{X_val.shape}  test:{X_test.shape}")
print(f"fraud -> train:{int(y_train.sum())}  val:{int(y_val.sum())}  test:{int(y_test.sum())}")

seq   -> train:(889904, 20, 82)  val:(112365, 20, 82)  test:(241909, 20, 82)
fraud -> train:4461  val:656  test:1238


## Model

A single LSTM layer over the 20-step window, followed by dropout and a sigmoid head.
Class imbalance (~0.5% fraud) is handled with **class weights** computed from the training
labels only.

In [8]:
pos, neg = int(y_train.sum()), int(len(y_train) - y_train.sum())
class_weight = {0: 1.0, 1: neg / max(pos, 1)}    # up-weight the rare fraud class
print("class_weight:", class_weight)

model = Sequential([
    LSTM(64, input_shape=(SEQUENCE_LENGTH, num_features), return_sequences=False),
    Dropout(0.2),
    Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

class_weight: {0: 1.0, 1: 198.4853171934544}


I0000 00:00:1785728203.551441      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        37,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,697 (147.25 KB)

 Trainable params: 37,697 (147.25 KB)

 Non-trainable params: 0 (0.00 B)

## Train

In [9]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30, batch_size=10000,
    class_weight=class_weight, verbose=1,
)

Epoch 1/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 14s 126ms/step - accuracy: 0.9133 - loss: 0.7123 - val_accuracy: 0.9518 - val_loss: 0.2477
Epoch 2/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 9s 104ms/step - accuracy: 0.9630 - loss: 0.3973 - val_accuracy: 0.9565 - val_loss: 0.1813
Epoch 3/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 9s 105ms/step - accuracy: 0.9636 - loss: 0.3305 - val_accuracy: 0.9511 - val_loss: 0.1790
Epoch 4/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 9s 104ms/step - accuracy: 0.9651 - loss: 0.2890 - val_accuracy: 0.9605 - val_loss: 0.1460
Epoch 5/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 9s 104ms/step - accuracy: 0.9676 - loss: 0.2563 - val_accuracy: 0.9582 - val_loss: 0.1445
Epoch 6/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 9s 105ms/step - accuracy: 0.9700 - loss: 0.2251 - val_accuracy: 0.9662 - val_loss: 0.1181
Epoch 7/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 9s 104ms/step - accuracy: 0.9721 - loss: 0.1959 - val_accuracy: 0.9674 - val_loss: 0.1105
Epoch 8/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 9s 104ms/step - accuracy: 0.9735 - loss: 0.1759 - val_accuracy: 0

## Evaluation

**PR-AUC** (average precision) is the primary metric because it is threshold-independent
and meaningful under heavy imbalance. The decision threshold is **tuned on validation**
(max-F1) and then applied *unchanged* to test.

In [10]:
def tune_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1 = np.where((prec + rec) > 0, 2 * prec * rec / (prec + rec), 0.0)
    return float(thr[np.argmax(f1[:-1])]) if len(thr) else 0.5


def report(y_true, y_prob, threshold, title):
    y_pred = (y_prob >= threshold).astype(int)
    print(f"\n=== {title} (threshold={threshold:.3f}) ===")
    print(f"PR-AUC   : {average_precision_score(y_true, y_prob):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1       : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(classification_report(
        y_true, y_pred, target_names=["Non-Fraud", "Fraud"], zero_division=0))

In [11]:
val_prob = model.predict(X_val).ravel()
threshold = tune_threshold(y_val, val_prob)
report(y_val, val_prob, threshold, "VALIDATION")

3512/3512 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step

=== VALIDATION (threshold=0.986) ===
PR-AUC   : 0.7598
Precision: 0.8304
Recall   : 0.6418
F1       : 0.7240
              precision    recall  f1-score   support

   Non-Fraud       1.00      1.00      1.00    111709
       Fraud       0.83      0.64      0.72       656

    accuracy                           1.00    112365
   macro avg       0.91      0.82      0.86    112365
weighted avg       1.00      1.00      1.00    112365



In [12]:
test_prob = model.predict(X_test).ravel()
report(y_test, test_prob, threshold, "TEST")   

7560/7560 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step

=== TEST (threshold=0.986) ===
PR-AUC   : 0.7393
Precision: 0.8412
Recall   : 0.6163
F1       : 0.7114
              precision    recall  f1-score   support

   Non-Fraud       1.00      1.00      1.00    240671
       Fraud       0.84      0.62      0.71      1238

    accuracy                           1.00    241909
   macro avg       0.92      0.81      0.86    241909
weighted avg       1.00      1.00      1.00    241909

